# Epoch Spring Camp 2026 - Take Home Task 3

In this task, you will build and compare two recommender system models:

- **Matrix Factorization (MF)** using a dot product of embeddings  
- **Neural Collaborative Filtering (MLP-based)** using a multi-layer perceptron  

You are provided with:
- Preprocessed interaction data
- Evaluation pipeline

You are expected to implement:
- Negative sampling
- Model architectures
- Training loop

---

The purpose of this task is to understand how neural networks can model user–item interactions beyond simple similarity.

## Imports

In [13]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Loading Data

We begin by loading the interaction data.

In [14]:
df = pd.read_csv("interactions.csv")  # columns: user_id, movie_id

print("Num users:", df['user_id'].nunique())
print("Num items:", df['item_id'].nunique())
print("Num interactions:", len(df))

Num users: 942
Num items: 1447
Num interactions: 55375


Each row represents a **positive interaction** (implicit feedback).

## Train / Test Split

We split the data into training and testing sets.

The model will learn from the training data and be evaluated on unseen interactions in the test set.

In [15]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

## Negative Sampling

The dataset only contains **positive interactions** (user interacted with item).

However, to train a model, we also need **negative examples** (user did not interact with item).

---

### Why do we need this?

We treat recommendation as a **binary classification problem**:

- Positive interaction → label = 1  
- No interaction → label = 0  

Since we don’t have explicit negatives, we **sample them**.

---

### Your Task

For each `(user, item)` interaction:
- Keep it as a **positive sample (label = 1)**
- Sample one or more items the user has **not interacted with**
  - Add them as **negative samples (label = 0)**

---

### Hints

- You can randomly sample items
- Make sure sampled negatives are **not already in the dataset**
- You may use a `set` of `(user, item)` pairs for fast lookup

---

Implement a function:
`sample_negatives(df, num_neg=1)`
that returns a dataframe with columns:
`[user, item, label]`

In [16]:
def sample_negatives(df, num_neg=1):
    # TODO:
    # 1. Create a set of (user, item) interactions
    # 2. For each positive interaction:
    #    - add (user, item, 1)
    #    - sample 'num_neg' negative items
    # 3. Return a dataframe with columns: user, item, label

    interactions = set(zip(df['user_id'], df['item_id']))
    all = df['item_id'].unique()
    r = []
    for _, row in df.iterrows():
        user, item = row['user_id'], row['item_id']
        r.append((user, item, 1))

        #sample num_neg negative items
        neg_count = 0
        while neg_count < num_neg:
            neg_item = np.random.choice(all)

            #makin sure it's not already a known interaction
            if (user, neg_item) not in interactions:
                r.append((user, neg_item, 0))
                neg_count += 1

    return pd.DataFrame(r, columns=['user_id', 'item_id', 'label'])

    pass

## PyTorch Dataset

We now wrap our processed data into a PyTorch `Dataset`.

This allows us to:
- Access individual samples as `(user, item, label)`
- Easily plug into a `DataLoader` for batching

You do not need to modify this part.

In [17]:
class InteractionDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['user_id'].values)
        self.items = torch.tensor(df['item_id'].values)
        self.labels = torch.tensor(df['label'].values).float()

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]

## DataLoader

The `DataLoader` will:
- Batch the data
- Shuffle the training data
- Feed it to the model during training

In [18]:
train_data = sample_negatives(train_df, num_neg = 15)

train_loader = DataLoader(InteractionDataset(train_data), batch_size=256, shuffle=True)

Why are we sampling negatives only for the training data?

## Quick Exploration

Before building models, take a moment to explore the data.

Try to understand:
- How many interactions each user has
- How popular certain items are

This can give intuition about the dataset.

In [19]:
# Interactions per user
user_counts = df['user_id'].value_counts()
print(user_counts.describe())

# Interactions per item
item_counts = df['item_id'].value_counts()
print(item_counts.describe())

count    942.000000
mean      58.784501
std       54.696664
min        3.000000
25%       19.000000
50%       39.500000
75%       80.750000
max      378.000000
Name: count, dtype: float64
count    1447.000000
mean       38.268832
std        57.956847
min         1.000000
25%         3.000000
50%        13.000000
75%        47.500000
max       501.000000
Name: count, dtype: float64


## Baseline Model: Matrix Factorization (MF)

We represent:
- Each **user** as a vector (embedding)
- Each **item** as a vector (embedding)

To predict interaction:
- Compute the **dot product** between user and item embeddings

This gives a **score** indicating how likely the user is to interact with the item.

---

### Your Task

Implement a model that:
1. Learns user and item embeddings
2. Computes their dot product as the output score

In [20]:
class MF(nn.Module):
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()

        # TODO:
        # - Define user embedding layer
        self.user_emb = nn.Embedding(num_users, emb_dim)
        # - Define item embedding layer
        self.item_emb = nn.Embedding(num_items, emb_dim)

    def forward(self, user, item):
        # TODO:
        # - Get user and item embeddings
        user_v = self.user_emb(user)
        item_v = self.item_emb(item)
        # - Compute dot product
        score = torch.sum(user_v * item_v, dim=1)
        # - Return a single score per pair
        return score
        pass

## Training the MF Model

Now train your Matrix Factorization model.

You will need to:
- Define a loss function
- Define an optimizer
- Iterate over the DataLoader
- Update model parameters

---

### Hints

- Use **Binary Cross Entropy loss**
- Apply **sigmoid** to model outputs if needed
- Typical steps:
  - forward pass
  - compute loss
  - backward pass
  - optimizer step

In [21]:
def train(model: nn.Module, dataloader: DataLoader, epochs: int):
    model = model.to('cuda' if torch.cuda.is_available() else 'cpu')
    model.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.BCEWithLogitsLoss()

    losses = []  # track loss per epoch

    for epoch in range(epochs):
        total_loss = 0

        for user, item, label in dataloader:
            user = user.to('cuda' if torch.cuda.is_available() else 'cpu')
            item = item.to('cuda' if torch.cuda.is_available() else 'cpu')
            label = label.to('cuda' if torch.cuda.is_available() else 'cpu')

            fwd = model.forward(user, item)
            loss = loss_fn(fwd, label)
            total_loss += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        losses.append(total_loss)
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

    return losses

## Evaluation (Hit@K)

We evaluate the model using a ranking-based metric.

For each user:
- Take one positive item
- Sample multiple negative items
- Rank all items using the model
- Check if the positive item is in the top-K

This is called **Hit@K**.

In [22]:
def hit_at_k(model, test_df, full_df, K=10, num_neg=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    model.to(device)

    hits = 0
    total = len(df) # We evaluate every positive interaction in the test set

    # Map interactions for quick lookup
    interacted_items = full_df.groupby('user_id')['item_id'].apply(set).to_dict()
    all_items = full_df['item_id'].unique()

    with torch.no_grad(): # Disable gradient calculation for speed/memory
        for _, row in df.iterrows():
            u = int(row['user_id'])
            pos_item = int(row['item_id'])

            # 1. Sample Negatives
            negatives = []
            while len(negatives) < num_neg:
                neg_item = np.random.choice(all_items)
                if neg_item not in interacted_items.get(u, set()):
                    negatives.append(neg_item)

            # 2. Prepare Tensors
            # We need a list of the 1 positive + 100 negatives
            item_list = [pos_item] + negatives
            user_tensor = torch.tensor([u] * (num_neg + 1)).to(device)
            item_tensor = torch.tensor(item_list).to(device)

            # 3. Get Scores
            scores = model(user_tensor, item_tensor)

            # 4. Rank and Check Hit
            # We want to see if the item at index 0 (the positive) is in the top K
            # torch.topk returns values and indices of the highest scores
            _, top_indices = torch.topk(scores, K)

            top_indices = top_indices.cpu().numpy()
            if 0 in top_indices:
                hits += 1

    return hits / total

# TODO:
# 1. Initialize MF model
# 2. Train it
# 3. Evaluate it

model = MF(num_users=df['user_id'].nunique(), num_items=df['item_id'].nunique(), emb_dim=1024)
losses = train(model, train_loader, epochs=40)
hit_rate = hit_at_k(model, test_df, df, K=10, num_neg=100)
print(f"\nHit@10: {hit_rate}")

#plotting loss
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Total Loss')
plt.title('Training Loss')
plt.show()

print(f"MF  Hit@10: {mf_hit:.4f}")
print(f"MLP Hit@10: {mlp_hit:.4f}")

Epoch 1, Loss: 31135.0226
Epoch 2, Loss: 16784.4177
Epoch 3, Loss: 7350.9353
Epoch 4, Loss: 3205.9607
Epoch 5, Loss: 1363.2163
Epoch 6, Loss: 478.8703
Epoch 7, Loss: 112.2244
Epoch 8, Loss: 12.7590
Epoch 9, Loss: 0.9811


KeyboardInterrupt: 

If your score is low, try playing with the hyperparameters before moving on! Try sampling more negatives, or playing with the embedding dimensions.

Conversely, if your score is high, play with the values of K or number of negatives in evaluation (dec K, inc negatives).

## Neural Model: MLP-Based Recommender

Instead of using a dot product, we can learn a more flexible interaction function using a neural network.

Approach:
- Learn user and item embeddings
- **Concatenate** them
- Pass through a **Multi-Layer Perceptron (MLP)**

This allows the model to capture more complex relationships than simple similarity.

---

### Your Task

Implement a model that:
1. Learns user and item embeddings
2. Concatenates them
3. Passes them through an MLP to produce a score

The choice of activation functions is upto you.

In [ ]:
class MLP(nn.Module):
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()

        # TODO:
        # - Define user embedding layer
        # - Define item embedding layer
        # - Define MLP layers

        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, user, item):
        # TODO:
        # - Get embeddings
        # - Concatenate user and item embeddings
        # - Pass through MLP
        # - Return a single score

        u = self.user_emb(user)
        i = self.item_emb(item)
        conc = torch.cat([u, i], dim=1)
        return self.mlp(conc).squeeze()

        pass



## Train and Evaluate the MLP Model

Repeat the same steps as before:
- Train the model
- Evaluate on the test set

Compare its performance with the MF model.

In [ ]:
# TODO:
# 1. Initialize MLP model
# 2. Train it
# 3. Evaluate it

MLP_model = MLP(num_users=df['user_id'].nunique(), num_items=df['item_id'].nunique(), emb_dim=128)
losses = train(MLP_model, train_loader, epochs=200)
hit_rate = hit_at_k(MLP_model, test_df, df, K=10, num_neg=100)
print(f"\nHit@10: {hit_rate:.4f}")


#plotting loss
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Total Loss')
plt.title('MLP Training Loss')
plt.show()

## Comparison & Analysis

You have now implemented:
- Matrix Factorization (MF)
- MLP-based recommender

---

### Compare the Models

- What score did each model achieve?
- Which model performed better?

---

### Think & Reflect

- Why might the MLP model outperform MF?
- In what cases might MF perform just as well or better?
- How does embedding size affect performance?
- Did you observe any signs of overfitting?

---

### Some Experiments

If you want to explore further:
- Try different embedding dimensions
- Change number of MLP layers
- Try different activation functions

---

## Bonus Task: Neural Collaborative Filtering (NCF)

In this task, you implemented:
- Matrix Factorization (MF)
- MLP-based recommender

The paper [Neural Collaborative Filtering](https://arxiv.org/pdf/1708.05031) proposes combining both ideas.

---

### Idea

- MF captures **linear interactions**
- MLP captures **nonlinear interactions**

NCF combines both by:
1. Computing MF output
2. Computing MLP output
3. Combining them into a final prediction

---

### Your Task

Design a model that:
- Uses both MF and MLP components
- Combines their outputs
- Trains end-to-end

---

### Hints

- You can:
  - Concatenate MF and MLP representations
  - Or combine their final scores
- Think about:
  - Should embeddings be shared or separate?
  - How to balance both components?

---

Does combining both approaches improve performance over MF and MLP individually?

In [ ]:
class NCF(nn.Module):
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()

        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

        #MLP path
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        #combining
        self.fusion = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, user, item):
        user_vec = self.user_emb(user)
        item_vec = self.item_emb(item)

        #MF output
        mf_score = torch.sum(user_vec * item_vec, dim=1)

        #MLP output
        mlp_score = self.mlp(torch.cat([user_vec, item_vec], dim=1)).squeeze()

        #combine
        combined = torch.stack([mf_score, mlp_score], dim=1)
        out = self.fusion(combined).squeeze()

        return out

In [ ]:
NCF_model = NCF(num_users=df['user_id'].nunique(), num_items=df['item_id'].nunique(), emb_dim=128)
losses = train(NCF_model, train_loader, epochs=200)
hit_rate = hit_at_k(NCF_model, test_df, df, K=10, num_neg=100)
print(f"\nHit@10: {hit_rate:.4f}")

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Total Loss')
plt.title('NCF Training Loss')
plt.show()